# Lesson 2 - Vector Search - Code Along 1 - Embeddings

In [31]:
# dependencies
import numpy as np
import json

from sentence_transformers import SentenceTransformer
from ingest import load_faq_data
from tqdm.auto import tqdm

In [2]:
# use a small model running locally for getting text embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
# encode a few examples to demonstrate model behavior
# first, encode a question q1
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [4]:
# encode the answer to the q1
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
# compare the question to the answer using dot product
v1.dot(dv)

np.float32(0.32332394)

In [6]:
# encode and compare an unrelated question
# the similarity with the document should be smaller
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
v2.dot(dv)

np.float32(0.019730562)

## Embedding the Dataset

In [7]:
# get the course FAQ dataset
documents = load_faq_data()

# check how many documents there are
print(len(documents))

1350


In [8]:
# generate embeddings for the FAQ dataset

# list for containing the embeddings
texts = []

# embedd question and answer together so a query can match both
for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [9]:
# divide the dataset into batches of 50 and encode each batch
batch_size = 50
vectors = []

# encode batches of 50 using the model and add them to the list
# tqdm for progress bar
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

# check result or rather see how many were embedded
# if each documents is embedded, it should be 1350
print(len(vectors))

  0%|          | 0/27 [00:00<?, ?it/s]

1350


It's 1350 vectors, just as the number of documents, so each one should be
embedded now.

In [10]:
# turn the embeddings into a matrix
# rows are documents, columns are dimensions of embeddings
X = np.array(vectors)

# shape should be 1350 x 384
# the small model we use has an output dim of 384
print(X.shape)

(1350, 384)


## Vector Search on the Embeddings

In [ ]:
# compute dot product of question embedding against all document embeddings
# dot product equals cosine similarity, because embedding space is L2 normalized
# v1 is the embedding of the question used in the beginning
# scroll up to see it
scores = X.dot(v1)

In [28]:
# get the best match -> highest cosine similarity -> highest score
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629408))

In [36]:
# see which Q&A pair it is
# expectation: we asked a literal question from the FAQ -> it will be that one
print("We asked:", q1, "\n") # the question we asked
print("Best match:\n", json.dumps(documents[idx], indent=2))

We asked: Can I still join the course after the start date? 

Best match:
 {
  "course": "data-engineering-zoomcamp",
  "section": "General Course-Related Questions",
  "question": "Course: Can I still join the course after the start date?",
  "answer": "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
  "doc_id": "3f1424af17"
}


In [41]:
# get the best 5 results, not just a single one
top5 = np.argsort(-scores)[:5]
top5

array([  2, 625, 907, 538,   7])

In [42]:
# check scores of these top results
scores[top5]

array([0.7629408 , 0.757937  , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

In [43]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629408
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.757937
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 